# Agent Middleware & Memory (Checkpointers) in LangGraph / LangChain

This notebook demonstrates how to create a ReAct agent with **Memory Checkpointers** using `ChatGroq` to track conversation history per thread.

### Step 1: Setup Tools & Model
We use `ChatGroq(model="llama-3.3-70b-versatile")` to avoid OpenAI 429 quota errors.

In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

load_dotenv()

# 1. Define Tools
@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

tools = [add, multiply]

# 2. Initialize Model
model = ChatGroq(model="llama-3.3-70b-versatile")

### Step 2: Create Agent with Memory Checkpointer

In [2]:
# Set up in-memory checkpointer
memory = MemorySaver()

# Create agent with tools and checkpointer
agent = create_react_agent(model, tools, checkpointer=memory)

### Step 3: Execute Queries in a Threaded Session

In [3]:
# Thread configuration for state persistence
config = {"configurable": {"thread_id": "1"}}

questions = [
    "What is 2+2?",
    "What is 4*4?",
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"--- Question: {q} ---")
    print(f"Response: {response['messages'][-1].content}")
    print(f"Total Messages in Thread: {len(response['messages'])}\n")

--- Question: What is 2+2? ---
Response: The answer is 4.
Total Messages in Thread: 4

--- Question: What is 4*4? ---
Response: The answer is 16.
Total Messages in Thread: 8
